# Research Question 7: Performance and Interpretability

This Kaggle-ready notebook loads the raw Electric Vehicle Population dataset, cleans it, computes the actual table for this research question, and saves the publication-ready figure as a PDF.

**Input:** raw CSV or Excel dataset attached to Kaggle.

**Outputs:** one CSV table and one PDF figure saved to `/kaggle/working`.

In [ ]:

# Common setup for Kaggle
# This notebook is self-contained. On Kaggle, attach the dataset and run all cells.

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COL = "Electric Vehicle Type"
POSITIVE_CLASS_NAME = "Battery Electric Vehicle (BEV)"
NEGATIVE_CLASS_NAME = "Plug-in Hybrid Electric Vehicle (PHEV)"

# On Kaggle, outputs should be written here. Locally, this falls back to ./outputs.
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional: set DATASET_PATH manually if automatic discovery does not find your file.
# Examples:
# DATASET_PATH = "/kaggle/input/electric-vehicle-population-data/Electric_Vehicle_Population_Data.csv"
# DATASET_PATH = "/kaggle/input/electric-vehicle-population-data/Electric_Vehicle_Population_Data.xlsx"
DATASET_PATH = None


def find_dataset_file():
    """Find the EV population dataset on Kaggle or in the current directory."""
    if DATASET_PATH and os.path.exists(DATASET_PATH):
        return DATASET_PATH
    patterns = [
        "/kaggle/input/**/Electric_Vehicle_Population_Data.csv",
        "/kaggle/input/**/Electric_Vehicle_Population_Data.xlsx",
        "/kaggle/input/**/*Electric*Vehicle*Population*.csv",
        "/kaggle/input/**/*Electric*Vehicle*Population*.xlsx",
        "./Electric_Vehicle_Population_Data.csv",
        "./Electric_Vehicle_Population_Data.xlsx",
        "../input/**/Electric_Vehicle_Population_Data.csv",
        "../input/**/Electric_Vehicle_Population_Data.xlsx",
    ]
    matches = []
    for pattern in patterns:
        matches.extend(glob.glob(pattern, recursive=True))
    matches = [m for m in matches if os.path.isfile(m)]
    if not matches:
        raise FileNotFoundError(
            "Could not find the dataset file. Set DATASET_PATH to the CSV or Excel file path."
        )
    return matches[0]


def load_ev_data():
    """Load CSV or Excel EV dataset."""
    path = find_dataset_file()
    print(f"Loading dataset from: {path}")
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    elif path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        raise ValueError("Unsupported file type. Use .csv, .xlsx, or .xls")
    print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df


def clean_ev_data(df):
    """Clean dataset and create binary target: 1 = BEV, 0 = PHEV."""
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    required = [TARGET_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep only BEV and PHEV rows.
    df = df[df[TARGET_COL].isin([POSITIVE_CLASS_NAME, NEGATIVE_CLASS_NAME])].copy()
    df["EV_Type_Binary"] = (df[TARGET_COL] == POSITIVE_CLASS_NAME).astype(int)

    # Numeric columns.
    for col in ["Model Year", "Electric Range", "Base MSRP"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Treat location/census identifiers as categorical strings, not continuous numbers.
    id_like_cols = ["Postal Code", "Legislative District", "2020 Census Tract"]
    for col in id_like_cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: "Unknown" if pd.isna(x) else str(int(x)) if isinstance(x, (int, float, np.integer, np.floating)) and float(x).is_integer() else str(x))

    # Fill categorical missing values.
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].fillna("Unknown").astype(str)

    return df


def available_features(df, feature_list):
    return [c for c in feature_list if c in df.columns]


VEHICLE_FEATURES = [
    "Model Year",
    "Make",
    "Model",
    "Clean Alternative Fuel Vehicle (CAFV) Eligibility",
    "Electric Range",
    "Base MSRP",
]

GEOGRAPHIC_FEATURES = [
    "County",
    "City",
    "Postal Code",
    "Legislative District",
    "Electric Utility",
    "2020 Census Tract",
]

ALL_FEATURES = VEHICLE_FEATURES + GEOGRAPHIC_FEATURES


def split_feature_types(df, features):
    numeric = [c for c in features if c in ["Model Year", "Electric Range", "Base MSRP"]]
    categorical = [c for c in features if c not in numeric]
    return numeric, categorical


def make_ohe():
    """Create OneHotEncoder compatible with multiple scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse=True)


def build_preprocessor(df, features, encoding="ordinal"):
    """Build preprocessing pipeline.
    encoding='ordinal' is fast and works well for tree-based models.
    encoding='onehot' is better for Logistic Regression.
    """
    features = available_features(df, features)
    numeric_features, categorical_features = split_feature_types(df, features)

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler() if encoding == "onehot" else "passthrough")
    ])

    if encoding == "onehot":
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("encoder", make_ohe())
        ])
    else:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )
    return preprocessor


def get_xy(df, features):
    features = available_features(df, features)
    X = df[features].copy()
    y = df["EV_Type_Binary"].copy()
    return X, y


def train_test_for_features(df, features):
    X, y = get_xy(df, features)
    return train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )


def evaluate_pipeline(pipe, X_test, y_test, model_name):
    y_pred = pipe.predict(X_test)
    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(X_test)[:, 1]
    elif hasattr(pipe, "decision_function"):
        y_score = pipe.decision_function(X_test)
    else:
        y_score = y_pred

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_score),
    }


def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path


def save_figure(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path


def set_publication_style():
    plt.rcParams.update({
        "figure.figsize": (9.5, 6.5),
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def add_bar_labels(ax, fmt="{:.3f}"):
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, padding=3, fontsize=9)


In [ ]:

# RQ7. Which machine learning model provides the best balance of predictive performance and interpretability?

set_publication_style()
df_raw = load_ev_data()
df = clean_ev_data(df_raw)
features = available_features(df, ALL_FEATURES)

X_train, X_test, y_train, y_test = train_test_for_features(df, features)

models = []
models.append(("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced"), "onehot", 5))
models.append(("Decision Tree", DecisionTreeClassifier(max_depth=18, min_samples_leaf=20, random_state=RANDOM_STATE, class_weight="balanced"), "ordinal", 4))
models.append(("Random Forest", RandomForestClassifier(n_estimators=150, min_samples_leaf=10, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1), "ordinal", 3))
models.append(("Gradient Boosting", GradientBoostingClassifier(random_state=RANDOM_STATE), "ordinal", 2))

try:
    from xgboost import XGBClassifier
    models.append(("XGBoost", XGBClassifier(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ), "ordinal", 2))
except Exception as e:
    print("XGBoost is not available; skipping it.", e)

rows = []
for model_name, estimator, encoding, interpretability in models:
    print(f"Training {model_name}...")
    pipe = Pipeline(steps=[
        ("preprocessor", build_preprocessor(df, features, encoding=encoding)),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    metrics = evaluate_pipeline(pipe, X_test, y_test, model_name)
    metrics["Interpretability score"] = interpretability
    rows.append(metrics)

rq7_table = pd.DataFrame(rows)
# Composite score: average of F1-score and normalized interpretability score.
rq7_table["Normalized interpretability"] = rq7_table["Interpretability score"] / 5
rq7_table["Balance score"] = (rq7_table["F1-score"] + rq7_table["Normalized interpretability"]) / 2
rq7_table["Overall assessment"] = np.where(
    rq7_table["Balance score"] == rq7_table["Balance score"].max(),
    "Best balance",
    "Comparison model"
)
rq7_table = rq7_table.sort_values("Balance score", ascending=False).reset_index(drop=True)

save_table(rq7_table, "RQ7_table_performance_interpretability_tradeoff.csv")
display(rq7_table)

# Figure 7: scatter plot of performance vs interpretability.
fig, ax = plt.subplots()
ax.scatter(rq7_table["Interpretability score"], rq7_table["F1-score"], s=100)
for _, row in rq7_table.iterrows():
    ax.annotate(row["Model"], (row["Interpretability score"], row["F1-score"]), xytext=(7, 5), textcoords="offset points")

mean_interp = rq7_table["Interpretability score"].mean()
mean_f1 = rq7_table["F1-score"].mean()
ax.axvline(mean_interp, linestyle="--", alpha=0.5)
ax.axhline(mean_f1, linestyle="--", alpha=0.5)
ax.text(mean_interp + 0.05, ax.get_ylim()[0] + 0.01, f"Mean interpretability = {mean_interp:.2f}", fontsize=9)
ax.text(ax.get_xlim()[0] + 0.05, mean_f1 + 0.005, f"Mean F1-score = {mean_f1:.3f}", fontsize=9)

ax.set_title("Figure 7. Trade-off between model performance and interpretability")
ax.set_xlabel("Interpretability score (1 = low, 5 = high)")
ax.set_ylabel("F1-score")
ax.set_xlim(1, 5.3)
ax.set_ylim(max(0, rq7_table["F1-score"].min() - 0.05), min(1.02, rq7_table["F1-score"].max() + 0.05))
ax.grid(linestyle="--", alpha=0.35)
plt.figtext(0.5, -0.04, "Note: Interpretability scores are assigned conceptually; F1-scores are computed from the test set.", ha="center", fontsize=10, style="italic")
save_figure("RQ7_figure_performance_interpretability_tradeoff.pdf")
plt.show()

best = rq7_table.iloc[0]
print(f"RQ7 answer: {best['Model']} provided the best balance with balance score {best['Balance score']:.3f}, F1-score {best['F1-score']:.3f}, and interpretability score {best['Interpretability score']}.")
